# EX_08 — Introducción a agentes (ejercicios)

**Notebook de referencia:** `notebook/08_Introduccion_Agentes.ipynb`

**Tiempo orientativo:** ~30 minutos.


## Actividad 1 — Definir 2 herramientas

Escribe funciones Python puras `get_time_utc()` (puede ser fake) y `hash_text(s: str)` (usa `hashlib.sha256` en hex). Estas serán tus "tools".


In [1]:
import hashlib
from datetime import datetime, timezone

def get_time_utc() -> str:
    """Devuelve la hora actual en UTC en formato ISO 8601."""
    return datetime.now(timezone.utc).isoformat()

def hash_text(s: str) -> str:
    """Calcula el digest SHA-256 de un texto y lo devuelve en hexadecimal."""
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

# --- Pruebas rápidas ---
print("UTC:", get_time_utc())
print("hash('hello'):", hash_text("hello"))
print("hash('hello') len:", len(hash_text("hello")))  # 64 chars hex


UTC: 2026-06-23T14:27:57.451552+00:00
hash('hello'): 2cf24dba5fb0a30e26e83b2ac5b9e29e1b161e5c1fa7425e73043362938b9824
hash('hello') len: 64


## Actividad 2 — Cuándo usar tool

Para cada intención del usuario (`"What time is it?"`, `"Digest of hello"`), escribe en comentarios si el LLM debería llamar tool o responder directo.


In [2]:
# Intención: "What time is it?"
# → LLM debe LLAMAR a la tool get_time_utc().
#   La hora actual no está en el conocimiento estático del modelo;
#   requiere datos externos en tiempo real.

# Intención: "Digest of hello"
# → LLM debe LLAMAR a la tool hash_text("hello").
#   El hash SHA-256 es determinista pero costoso de calcular mentalmente;
#   la tool garantiza exactitud.

# Regla práctica: usar tool cuando la respuesta depende de
# (a) datos dinámicos (hora, APIs, BD) o (b) cómputo exacto/reproducible (hash, stats).
# Responder directo cuando basta el razonamiento lingüístico o conocimiento general estable.


## Actividad 3 — Bucles

En español (celda markdown), explica el riesgo de **bucles infinitos** tool→modelo→tool y una mitigación (límite de pasos, detector de repetición).


**Riesgo de bucles infinitos (tool → modelo → tool):**

En un agente, el LLM puede volver a invocar la misma herramienta con los mismos argumentos si no interpreta bien la observación anterior, o alternar entre dos tools sin converger (p. ej. `get_time_utc` repetido porque el prompt pide "hora exacta al milisegundo").

**Mitigaciones:**

1. **Límite de pasos (`max_iterations`)** — como en `AgentExecutor(max_iterations=5)`: corta el bucle tras N ciclos y devuelve error controlado.
2. **Detector de repetición** — guardar historial de `(tool_name, args)`; si se repite la misma acción dos veces seguidas sin cambio de estado, forzar respuesta final o escalar a humano.
3. **Condición de parada explícita** — instruir al modelo en el system prompt: *"Tras obtener la observación, responde al usuario; no vuelvas a llamar tools salvo error evidente."*
